In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/used_cars.csv')
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [3]:
# 1. Dataset Overview
df.shape
print(f'No Of Rows are {df.shape[0]}')
print(f'No Of Cols are {df.shape[1]}')
print(df.duplicated().sum())
total_byte = df.memory_usage(deep=True).sum()
total_mb = total_byte / (1024 ** 2)
print(f"Total memory usage: {total_mb:.2f} MB")

No Of Rows are 4009
No Of Cols are 12
0
Total memory usage: 2.58 MB


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB


In [5]:
# Step 2 — Column Type Inventory
# Numerical Columns
numerical_col = df.select_dtypes(include=[np.number]).columns
print(numerical_col)

Index(['model_year'], dtype='str')


In [6]:
# Fixing the data type of the data
import re
_SYMBOL_PATTERN = re.compile(
    r"""
        ^\s*
        [₹$€£¥₩%+\-]
        [\s]?
        |
        \s*
        (mi\.?|km\.?|kg\.?|lbs?\.?|mph|kph|hp|cc|°[cf]?|%|[₹$€£¥₩])
        \s*$
    """,
    re.IGNORECASE | re.VERBOSE
)
_BOOL_MAP = {
    "yes": 1, "no": 0,
    "true": 1, "false": 0,
    "on": 1, "off": 0,
    "on-off": 0, "true-on": 0, "false-off": 0, "false-on": 0, "true-off": 0,
    "y": 1, "n": 1,
    "1": 1, "0": 0
}

In [8]:
def _strip_to_numeric(value: str) -> str:
    """Remove currency symbols, units, commas from a string value."""
    v = _SYMBOL_PATTERN.sub("", str(value))  # .sub() method is used to find anything matching the prefixes or suffixes and replace with ""
    v = v.replace(",", "")   # remove thousand separators
    return v.strip()

In [9]:
def _is_dirty_numeric(series: pd.Series, sample_size: int=50) -> bool:
    sample = series.dropna().head(sample_size)
    if(sample) == 0:
        return False
    success = 0
    for val in sample:
        try:
            float(_strip_to_numeric(str(val)))
            success += 1
        except ValueError:
            pass
    return (success / len(sample)) >= 0.90
